In [1]:
import pandas as pd
from Bio import SeqIO
import re
import os
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import traceback

# --- CẤU HÌNH ---
FASTA_PATH = r"D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa"

WINDOW = 50 
TARGET_LEN = 101 # 50 + 1 (đột biến) + 50
PAD_CHAR = 'X'

# --- HÀM TIỆN ÍCH ---

def load_ensembl_protein_fasta(path):
    """
    Kế thừa logic load genome: Map từ Protein ID (ENSP) sang Protein Sequence.
    """
    mapping = {}
    print(f"🧬 Đang nạp FASTA từ {path}...")
    for record in SeqIO.parse(path, "fasta"):
        # Trích xuất ENST từ header: ... transcript:ENST00000641515.2 ...
        match = re.search(r'transcript:(ENST\d+)', record.description)
        if match:
            enst_id = match.group(1)
            mapping[enst_id] = str(record.seq).upper()
    print(f"✅ Đã nạp {len(mapping)} mã transcript.")
    return mapping

def normalize_centered_protein(seq, center_idx, target_len, pad_char='X'):
    """
    Kế thừa hoàn toàn logic 'Symmetric Crop/Pad' từ notebook DNA của bạn.
    """
    half = target_len // 2
    start = center_idx - half
    end = center_idx + half + 1
    
    pad_left = max(0, -start)
    pad_right = max(0, end - len(seq))
    
    crop_left = max(0, start)
    crop_right = min(len(seq), end)
    
    final_seq = (pad_char * pad_left) + seq[crop_left:crop_right] + (pad_char * pad_right)
    return final_seq[:target_len] # Đảm bảo luôn đủ 101

In [2]:
PARQUET_PATH = r"D:\variant_data\train1_final.parquet"
OUTPUT_PATH = r"D:\variant_data\train1_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\train1_final.parquet
 > Đã xử lý xong chunk 1...
 > Đã xử lý xong chunk 2...
🎉 Hoàn thành! File kết quả: D:\variant_data\train1_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 104,385

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ENSP,HGVSc,HGVSp,Review_Rank,Split,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,1_1014042_G_A,446939.0,single nucleotide variant,NM_005101.4(ISG15):c.62G>A (p.Ser21Asn),9636.0,ISG15,HGNC:4053,Benign/Likely benign,0.0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",...,ENSP00000496832,ENST00000649529.1:c.62G>A,ENSP00000496832.1:p.Ser21Asn,3.0,Train,1014042,GGTGGGGCACAGAGGGGCACCCTAGCAGGTAAAGGGAGGCCACGGG...,GGTGGGGCACAGAGGGGCACCCTAGCAGGTAAAGGGAGGCCACGGG...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMGWDLTVKMLAGNEFQ...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMGWDLTVKMLAGNEFQ...
1,1_1014228_G_A,389314.0,single nucleotide variant,NM_005101.4(ISG15):c.248G>A (p.Ser83Asn),9636.0,ISG15,HGNC:4053,Benign,0.0,"MedGen:CN169374|MONDO:MONDO:0014502,MedGen:C40...",...,ENSP00000496832,ENST00000649529.1:c.248G>A,ENSP00000496832.1:p.Ser83Asn,3.0,Train,1014228,TGTGTGGTGGGCCTGGGGCTGGCGCCGCAGTCTCTGAACCTGTGTG...,TGTGTGGTGGGCCTGGGGCTGGCGCCGCAGTCTCTGAACCTGTGTG...,TQKIGVHAFQQRLAVHPSGVALQDRVPLASQGLGPGSTVLLVVDKC...,TQKIGVHAFQQRLAVHPSGVALQDRVPLASQGLGPGSTVLLVVDKC...
2,1_1014401_G_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSP00000496832,ENST00000649529.1:c.421G>A,ENSP00000496832.1:p.Gly141Ser,0.0,Train,1014401,TTCCAGCAGCGTCTGGCTGTCCACCCGAGCGGTGTGGCGCTGCAGG...,TTCCAGCAGCGTCTGGCTGTCCACCCGAGCGGTGTGGCGCTGCAGG...,GRSSTYEVRLTQTVAHLKQQVSGLEGVQDDLFWLTFEGKPLEDQLP...,GRSSTYEVRLTQTVAHLKQQVSGLEGVQDDLFWLTFEGKPLEDQLP...
3,1_1014471_G_C,446981.0,single nucleotide variant,NM_005101.4(ISG15):c.491G>C (p.Arg164Pro),9636.0,ISG15,HGNC:4053,Likely benign,0.0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",...,ENSP00000496832,ENST00000649529.1:c.491G>C,ENSP00000496832.1:p.Arg164Pro,2.0,Train,1014471,GCCTGGGCCCCGGCAGCACGGTCCTGCTGGTGGTGGACAAATGCGA...,GCCTGGGCCCCGGCAGCACGGTCCTGCTGGTGGTGGACAAATGCGA...,LEGVQDDLFWLTFEGKPLEDQLPLGEYGLKPLSTVFMNLRLRGGGT...,LEGVQDDLFWLTFEGKPLEDQLPLGEYGLKPLSTVFMNLRLRGGGT...
4,1_1020183_G_C,364282.0,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790.0,AGRN,HGNC:329,Benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,ENSP00000368678,ENST00000379370.7:c.11G>C,ENSP00000368678.2:p.Arg4Pro,3.0,Train,1020183,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104380,X_155511709_C_T,2821775.0,single nucleotide variant,NM_018196.4(TMLHE):c.722G>A (p.Arg241Gln),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:C3661900,...,ENSP00000335261,ENST00000334398.8:c.722G>A,ENSP00000335261.3:p.Arg241Gln,2.0,Train,155511709,GAGACAATTAAATCTCTCTTTTTTTTTTTTATAAATTACCCAGTCT...,GAGACAATTAAATCTCTCTTTTTTTTTTTTATAAATTACCCAGTCT...,FVENVPPTQEHTEKLAERISLIRETIYGRMWYFTSDFSRGDTAYTK...,FVENVPPTQEHTEKLAERISLIRETIYGRMWYFTSDFSRGDTAYTK...
104381,X_155524537_G_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSP00000335261,ENST00000334398.8:c.277C>T,ENSP00000335261.3:p.Arg93Cys,0.0,Train,155524537,GGGAAAGCATTAAGTCTCACTATTAAGTATAATGTTAGCTGTGGAT...,GGGAAAGCATTAAGTCTCACTATTAAGTATAATGTTAGCTGTGGAT...,HTASKSLTCAWQQHEDHFELKYANTVMRFDYVWLRDHCRSASCYNS...,HTASKSLTCAWQQHEDHFELKYANTVMRFDYVWLRDHCRSASCYNS...
104382,X_155545128_G_T,2703929.0,single nucleotide variant,NM_018196.4(TMLHE):c.149C>A (p.Thr50Asn),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:CN169374,...,ENSP00000335261,ENST00000334398.8:c.149C>A,ENSP00000335261.3:p.Thr50Asn,2.0,Train,155545128,TGTCTTTCTCCCAACTTGTAGGCTAATCTTAAAAATAAATGGCTAG...,TGTCTTTCTCCCAACTTGTAGGCTAATCTTAAAAATAAATGGCTAG...,XMWYHRLSHLHSRLQDLLKGGVIYPALPQPNFKSLLPLAVHWHHTA...,XMWYHRLSHLHSRLQDLLKGGVIYPALPQPNFKSLLPLAVHWHHTA...
104383,X_155545219_C_T,3343405.0,single nucleotide variant,NM_018196.4(TMLHE):c.58G>A (p.Gly20Arg),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:CN169374,...,ENSP00000335261,ENST00000334

In [3]:
PARQUET_PATH = r"D:\variant_data\train2_final.parquet"
OUTPUT_PATH = r"D:\variant_data\train2_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\train2_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\train2_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 67,990

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ENSP,HGVSc,HGVSp,Review_Rank,Split,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,1_1020183_G_C,364282.0,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790.0,AGRN,HGNC:329,Benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,ENSP00000368678,ENST00000379370.7:c.11G>C,ENSP00000368678.2:p.Arg4Pro,3.0,Train,1020183,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,1_1022383_C_G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSP00000368678,ENST00000379370.7:c.384C>G,ENSP00000368678.2:p.His128Gln,0.0,Train,1022383,CACATCTCTGCCCAGGGCTTGAGTCTACTGTGGACATTTGCCCTAA...,CACATCTCTGCCCAGGGCTTGAGTCTACTGTGGACATTTGCCCTAA...,DLVARESLLDGGNKVVISGFGDPLICDNQVSTGDTRIFFVNPAPPY...,DLVARESLLDGGNKVVISGFGDPLICDNQVSTGDTRIFFVNPAPPY...
2,1_1035307_C_T,446942.0,single nucleotide variant,NM_198576.4(AGRN):c.494C>T (p.Pro165Leu),375790.0,AGRN,HGNC:329,Likely benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,ENSP00000368678,ENST00000379370.7:c.494C>T,ENSP00000368678.2:p.Pro165Leu,3.0,Train,1035307,TCCTGCCTGCACCCCTGTGGCTGGGGCCCCATCTGACAGGGGTCAG...,TCCTGCCTGCACCCCTGTGGCTGGGGCCCCATCTGACAGGGGTCAG...,FFVNPAPPYLWPAHKNELMLNSSLMRITLRNLEEVEFCVEDKPGTH...,FFVNPAPPYLWPAHKNELMLNSSLMRITLRNLEEVEFCVEDKPGTH...
3,1_1041218_C_T,249307.0,single nucleotide variant,NM_198576.4(AGRN):c.773C>T (p.Thr258Ile),375790.0,AGRN,HGNC:329,Benign/Likely benign,0.0,"MedGen:CN169374|MONDO:MONDO:0014052,MedGen:C38...",...,ENSP00000368678,ENST00000379370.7:c.773C>T,ENSP00000368678.2:p.Thr258Ile,3.0,Train,1041218,GCGGGGCCTATGAGATGGAGCGAGGCTGGGAGGGGCTTCGGGGCCA...,GCGGGGCCTATGAGATGGAGCGAGGCTGGGAGGGGCTTCGGGGCCA...,PVCGSDASTYSNECELQRAQCSQQRRIRLLSRGPCGSRDPCSNVTC...,PVCGSDASTYSNECELQRAQCSQQRRIRLLSRGPCGSRDPCSNVTC...
4,1_1041583_A_G,133740.0,single nucleotide variant,NM_198576.4(AGRN):c.1058A>G (p.Gln353Arg),375790.0,AGRN,HGNC:329,Benign,0.0,MedGen:CN169374|MedGen:C3661900|MONDO:MONDO:00...,...,ENSP00000368678,ENST00000379370.7:c.1058A>G,ENSP00000368678.2:p.Gln353Arg,3.0,Train,1041583,CCCGAGGGGACCGTCTGCGGCAGCGACGGCGCCGACTACCCCGGCG...,CCCGAGGGGACCGTCTGCGGCAGCGACGGCGCCGACTACCCCGGCG...,CARQENVFKKFDGPCDPCQGALPDPSRSCRVNPRTRRPEMLLRPES...,CARQENVFKKFDGPCDPCQGALPDPSRSCRVNPRTRRPEMLLRPES...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67985,X_155506983_C_T,4308127.0,single nucleotide variant,NM_018196.4(TMLHE):c.910G>A (p.Asp304Asn),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:CN169374,...,ENSP00000335261,ENST00000334398.8:c.910G>A,ENSP00000335261.3:p.Asp304Asn,2.0,Train,155506983,ACCCCACCCCTAAAATAAACCTAGTATTCTTTGGTTGGTGGCTATT...,ACCCCACCCCTAAAATAAACCTAGTATTCTTTGGTTGGTGGCTATT...,IQVFHCLKHEGTGGRTLLVDGFYAAEQVLQKAPEEFELLSKVPLKH...,IQVFHCLKHEGTGGRTLLVDGFYAAEQVLQKAPEEFELLSKVPLKH...
67986,X_155511709_C_T,2821775.0,single nucleotide variant,NM_018196.4(TMLHE):c.722G>A (p.Arg241Gln),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:C3661900,...,ENSP00000335261,ENST00000334398.8:c.722G>A,ENSP00000335261.3:p.Arg241Gln,2.0,Train,155511709,GAGACAATTAAATCTCTCTTTTTTTTTTTTATAAATTACCCAGTCT...,GAGACAATTAAATCTCTCTTTTTTTTTTTTATAAATTACCCAGTCT...,FVENVPPTQEHTEKLAERISLIRETIYGRMWYFTSDFSRGDTAYTK...,FVENVPPTQEHTEKLAERISLIRETIYGRMWYFTSDFSRGDTAYTK...
67987,X_155524537_G_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSP00000335261,ENST00000334398.8:c.277C>T,ENSP00000335261.3:p.Arg93Cys,0.0,Train,155524537,GGGAAAGCATTAAGTCTCACTATTAAGTATAATGTTAGCTGTGGAT...,GGGAAAGCATTAAGTCTCACTATTAAGTATAATGTTAGCTGTGGAT...,HTASKSLTCAWQQHEDHFELKYANTVMRFDYVWLRDHCRSASCYNS...,HTASKSLTCAWQQHEDHFELKYANTVMRFDYVWLRDHCRSASCYNS...
67988,X_155545128_G_T,2703929.0,single nucleotide variant,NM_018196.4(TMLHE):c.149C>A (p.Thr50Asn),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:CN169374,...,ENSP00000335261,ENST000003

In [4]:
PARQUET_PATH = r"D:\variant_data\train3_final.parquet"
OUTPUT_PATH = r"D:\variant_data\train3_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\train3_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\train3_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 23,208

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ENSP,HGVSc,HGVSp,Review_Rank,Split,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,12_53315378_C_T,2048536.0,single nucleotide variant,NM_015665.6(AAAS):c.356G>A (p.Arg119Gln),8086.0,AAAS,HGNC:13666,Likely benign,0.0,MedGen:C3661900,...,ENSP00000209873,ENST00000209873.9:c.356G>A,ENSP00000209873.4:p.Arg119Gln,3.0,Train,53315378,GGGCTTGTGCACTCACCAATTTGTGACTTGGGCAAATTCAGCGATC...,GGGCTTGTGCACTCACCAATTTGTGACTTGGGCAAATTCAGCGATC...,FIHHREQVWKRCINIWRDVGLFGVLNEIANSEEEVFEWVKTASGWA...,FIHHREQVWKRCINIWRDVGLFGVLNEIANSEEEVFEWVKTASGWA...
1,12_53321403_G_C,325881.0,single nucleotide variant,NM_015665.6(AAAS):c.63C>G (p.His21Gln),8086.0,AAAS,HGNC:13666,Likely benign,0.0,MedGen:C3661900|,...,ENSP00000209873,ENST00000209873.9:c.63C>G,ENSP00000209873.4:p.His21Gln,2.0,Train,53321403,ATTCTCCCTCATTCTCTCCATTCCCTCTAGGCCTCCTCCACAGCTC...,ATTCTCCCTCATTCTCTCCATTCCCTCTAGGCCTCCTCCACAGCTC...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPPPPRGQV...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPPPPRGQV...
2,12_53314832_C_T,859992.0,single nucleotide variant,NM_015665.6(AAAS):c.464G>A (p.Arg155His),8086.0,AAAS,HGNC:13666,Pathogenic/Likely pathogenic,1.0,"MedGen:C3661900|MONDO:MONDO:0009279,MedGen:C02...",...,ENSP00000209873,ENST00000209873.9:c.464G>A,ENSP00000209873.4:p.Arg155His,3.0,Train,53314832,GAAGGATGATCCATCCAGGGGCCAGGGGCACCATAGGACAGGAGGG...,GAAGGATGATCCATCCAGGGGCCAGGGGCACCATAGGACAGGAGGG...,EWVKTASGWALALCRWASSLHGSLFPHLSLRSEDLIAEFAQVTNWS...,EWVKTASGWALALCRWASSLHGSLFPHLSLRSEDLIAEFAQVTNWS...
3,12_53321423_G_T,20083.0,single nucleotide variant,NM_015665.6(AAAS):c.43C>A (p.Gln15Lys),8086.0,AAAS,HGNC:13666,Pathogenic,1.0,"MONDO:MONDO:0009279,MedGen:C0271742,OMIM:23155...",...,ENSP00000209873,ENST00000209873.9:c.43C>A,ENSP00000209873.4:p.Gln15Lys,3.0,Train,53321423,TTCCCTCTAGGCCTCCTCCACAGCTCCTTAACCGTTCCCCCGTTAT...,TTCCCTCTAGGCCTCCTCCACAGCTCCTTAACCGTTCCCCCGTTAT...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPP...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPP...
4,16_70258214_C_T,466634.0,single nucleotide variant,NM_001605.3(AARS1):c.1996G>A (p.Val666Ile),16.0,AARS1,HGNC:20,Likely benign,0.0,"MONDO:MONDO:0018993,MedGen:C0270914,Orphanet:6...",...,ENSP00000261772,ENST00000261772.13:c.1996G>A,ENSP00000261772.8:p.Val666Ile,3.0,Train,70258214,GGCATTTCTGAGGACTTCACTTCAACCTTAGACAGACAAGAGTTGG...,GGCATTTCTGAGGACTTCACTTCAACCTTAGACAGACAAGAGTTGG...,RSVLGEADQKGSLVAPDRLRFDFTAKGAMSTQQIKKAEEIANEMIE...,RSVLGEADQKGSLVAPDRLRFDFTAKGAMSTQQIKKAEEIANEMIE...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23203,7_76425104_A_G,3409690.0,single nucleotide variant,NM_001110354.2(ZP3):c.140A>G (p.Gln47Arg),7784.0,ZP3,HGNC:13189,Likely benign,0.0,MedGen:C3661900|MedGen:CN169374,...,ENSP00000378326,ENST00000394857.8:c.140A>G,ENSP00000378326.3:p.Gln47Arg,3.0,Train,76425104,AGGTGTTACTGATGCTTCTGGATGGAGACCACTTTATGCAACAAGG...,AGGTGTTACTGATGCTTCTGGATGGAGACCACTTTATGCAACAAGG...,XXXXMELSYRLFICLLLWGSTELCYPQPLWLLQGGASHPETSVQPV...,XXXXMELSYRLFICLLLWGSTELCYPQPLWLLQGGASHPETSVQPV...
23204,7_76429602_G_A,431554.0,single nucleotide variant,NM_001110354.2(ZP3):c.400G>A (p.Ala134Thr),7784.0,ZP3,HGNC:13189,Pathogenic/Likely pathogenic,1.0,"MONDO:MONDO:0021574,MedGen:C4540205,OMIM:61771...",...,ENSP00000378326,ENST00000394857.8:c.400G>A,ENSP00000378326.3:p.Ala134Thr,0.0,Train,76429602,GCTATGTTTCCCAGGCCAGTCTCAAACTCCTGGCCTCAAGCAATCC...,GCTATGTTTCCCAGGCCAGTCTCAAACTCCTGGCCTCAAGCAATCC...,MDTEDVVRFEVGLHECGNSMQVTDDALVYSTFLLHDPRPVGNLSIV...,MDTEDVVRFEVGLHECGNSMQVTDDALVYSTFLLHDPRPVGNLSIV...
23205,7_76434087_C_G,676995.0,single nucleotide variant,NM_001110354.2(ZP3):c.763C>G (p.Arg255Gly),7784.0,ZP3,HGNC:13189,Pathogenic,1.0,"MONDO:MONDO:0021574,MedGen:C4540205,OMIM:617712",...,ENSP00000378326,ENST00000394857.8:c.763C>G,ENSP00000378326.3:p.Arg255Gly,0.0,Train,76434087,CTCCCAGGTTTAAGTGATTCTCCTGCCTCAGCCCCCCAAGTAGCTG...,CTCCCAGGTTTAAGTGATTCTCCTGCCTCAGCCCCCCAAGTAGCTG...,GSHVPLRLFV

In [5]:
PARQUET_PATH = r"D:\variant_data\val_final.parquet"
OUTPUT_PATH = r"D:\variant_data\val_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\val_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\val_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 13,701

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ENSP,HGVSc,HGVSp,Review_Rank,Split,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,4_1022218_C_T,720682.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.95C>T (p.Ala32Val),53834.0,FGFRL1,HGNC:3693,Benign,0.0,MedGen:C3661900|,...,ENSP00000425025,ENST00000510644.6:c.95C>T,ENSP00000425025.1:p.Ala32Val,2.0,Val,1022218,CTCGTGTCTCAAAGAGCCAGCCTCTGGGGCCACGGGGCTGCCCCGG...,CTCGTGTCTCAAAGAGCCAGCCTCTGGGGCCACGGGGCTGCCCCGG...,XXXXXXXXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARG...,XXXXXXXXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARG...
1,4_1022236_G_A,1949144.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.113G>A (p.Arg38Gln),53834.0,FGFRL1,HGNC:3693,Likely benign,0.0,MedGen:C3661900,...,ENSP00000425025,ENST00000510644.6:c.113G>A,ENSP00000425025.1:p.Arg38Gln,2.0,Val,1022236,AGCCTCTGGGGCCACGGGGCTGCCCCGGCCATGAGAGGCTGCTGAC...,AGCCTCTGGGGCCACGGGGCTGCCCCGGCCATGAGAGGCTGCTGAC...,XXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARGPPKMAD...,XXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARGPPKMAD...
2,4_1023924_G_A,734353.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.541G>A (p.Asp181Asn),53834.0,FGFRL1,HGNC:3693,Benign/Likely benign,0.0,"MedGen:C3661900|MONDO:MONDO:0008684,MedGen:C19...",...,ENSP00000425025,ENST00000510644.6:c.541G>A,ENSP00000425025.1:p.Asp181Asn,3.0,Val,1023924,CCTCCGTCTCTCTGCAGATGACATTAGCCCAGGGAAGGAGAGCCTG...,CCTCCGTCTCTCTGCAGATGACATTAGCCCAGGGAAGGAGAGCCTG...,SSSGGQEDPASQQWARPRFTQPSKMRRRVIARPVGSSVRLKCVASG...,SSSGGQEDPASQQWARPRFTQPSKMRRRVIARPVGSSVRLKCVASG...
3,4_1024917_C_A,1154604.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.1085C>A (p.Pro362Gln),53834.0,FGFRL1,HGNC:3693,Benign,0.0,MedGen:C3661900,...,ENSP00000425025,ENST00000510644.6:c.1085C>A,ENSP00000425025.1:p.Pro362Gln,3.0,Val,1024917,ACACCATGGGCTACAGCTTCCGCAGCGCCTTCCTCACCGTGCTGCC...,ACACCATGGGCTACAGCTTCCGCAGCGCCTTCCTCACCGTGCTGCC...,WSRPDGSYLNKLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLP...,WSRPDGSYLNKLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLP...
4,4_1024946_G_A,2724602.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.1114G>A (p.Ala372Thr),53834.0,FGFRL1,HGNC:3693,Likely benign,0.0,MedGen:CN169374,...,ENSP00000425025,ENST00000510644.6:c.1114G>A,ENSP00000425025.1:p.Ala372Thr,2.0,Val,1024946,TTCCTCACCGTGCTGCCAGGTGCGCGGCTGCCACGCCACGCCACAC...,TTCCTCACCGTGCTGCCAGGTGCGCGGCTGCCACGCCACGCCACAC...,KLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLPDPKPPGPPVA...,KLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLPDPKPPGPPVA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13696,22_50744150_G_A,3640163.0,single nucleotide variant,NM_001097.3(ACR):c.655G>A (p.Val219Ile),49.0,ACR,HGNC:126,Likely benign,0.0,MedGen:CN169374,...,ENSP00000216139,ENST00000216139.10:c.655G>A,ENSP00000216139.5:p.Val219Ile,2.0,Val,50744150,TCCTCTGGCCTTGATTTGGATAACATTTCCCCACCTCCTCCCACCA...,TCCTCTGGCCTTGATTTGGATAACATTTCCCCACCTCCTCCCACCA...,GLPRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQW...,GLPRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQW...
13697,22_50744156_C_T,3640166.0,single nucleotide variant,NM_001097.3(ACR):c.661C>T (p.Pro221Ser),49.0,ACR,HGNC:126,Likely benign,0.0,MedGen:CN169374,...,ENSP00000216139,ENST00000216139.10:c.661C>T,ENSP00000216139.5:p.Pro221Ser,2.0,Val,50744156,GGCCTTGATTTGGATAACATTTCCCCACCTCCTCCCACCACCTCTA...,GGCCTTGATTTGGATAACATTTCCCCACCTCCTCCCACCACCTCTA...,PRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQWYN...,PRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQWYN...
13698,22_50744825_G_A,2754629.0,single nucleotide variant,NM_001097.3(ACR):c.884G>A (p.Arg295His),49.0,ACR,HGNC:126,Likely benign,0.0,MedGen:CN169374,...,ENSP00000216139,ENST00000216139.10:c.884G>A,ENSP00000216139.5:p.Arg295His,2.0,Val,50744825,AACCACTTGTGTTTACAGCAGCAGGAGACCATGTCACTGTGGAAAT...,AACCACTTGTGTTTACAGCAGCAGGAGACCATGTCACTGTGGAAAT...,MCKDSKESAYVVVGITSWGVGCARAKRPGIYTATWPYLNWIASKIG...,MCKDSKESAYVVVGITSWGVGCARAKRPGIYTATWPYLNWIASKIG...
13699,22_50744827_A_G,390380.0,single nucleotide variant,NM_001097.3(ACR):c.886A>G (p.Met296Val),

In [6]:
PARQUET_PATH = r"D:\variant_data\test_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\test_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\test_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\test_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 6,095

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,10_180088_C_T,424656.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.76C>T (p.Arg26Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic/Likely pathogenic,1.0,"MONDO:MONDO:0014486,MedGen:C4015167,OMIM:61608...",...,D,D,D,T,T,180088,AATGTCATTCTTCATGATGTAATGAAATGAATGATACTTTATATGA...,AATGTCATTCTTCATGATGTAATGAAATGAATGATACTTTATATGA...,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...
1,10_237638_G_T,2038412.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.570G>T (p.Arg190Ser),10771.0,ZMYND11,HGNC:16966,Likely benign,0.0,"MedGen:C3661900|MeSH:D030342,MedGen:C0950123",...,D,D,T,D,T,237638,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...
2,10_240913_C_G,2077177.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.774C>G (p.Cys258Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic,1.0,MedGen:C3661900,...,D,D,D,D,T,240913,TCTGAAGTGCTAACCAGTGAGGTCCGGGCAGGGCCCAGTCAGACCA...,TCTGAAGTGCTAACCAGTGAGGTCCGGGCAGGGCCCAGTCAGACCA...,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...
3,10_242031_A_C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,D,D,D,D,242031,GCTGGCTCCCCAGCTGCACTTGGCCAAGCGGGTGCTCTTGTTTGCC...,GCTGGCTCCCCAGCTGCACTTGGCCAAGCGGGTGCTCTTGTTTGCC...,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...
4,10_246823_C_G,3002871.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.1008C>G (p.His336Gln),10771.0,ZMYND11,HGNC:16966,Benign,0.0,MedGen:C3661900,...,T,D,T,D,T,246823,CAAATCTCATTTATTTTCCGCTTGGTAACAGTTTATTTATTCAAGC...,CAAATCTCATTTATTTTCCGCTTGGTAACAGTTTATTTATTCAAGC...,AKMKGFGFWPAKVMQKEDNQVDVRFFGHHHQRAWIPSENIQDITVN...,AKMKGFGFWPAKVMQKEDNQVDVRFFGHHHQRAWIPSENIQDITVN...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6090,20_64208061_G_A,728821.0,single nucleotide variant,NM_004535.3(MYT1):c.865G>A (p.Glu289Lys),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900|,...,T,T,T,T,T,64208061,AAGCCAGGTCCTGGCATTGTGCACCTGCTTCAGGAGGCTGCAGAGG...,AAGCCAGGTCCTGGCATTGTGCACCTGCTTCAGGAGGCTGCAGAGG...,SLEDAASEESSKQKGILSHEEEDEEEEEEEEEEEEDEEEEEEEEEE...,SLEDAASEESSKQKGILSHEEEDEEEEEEEEEEEEDEEEEEEEEEE...
6091,20_64208182_G_A,4228399.0,single nucleotide variant,NM_004535.3(MYT1):c.986G>A (p.Arg329Gln),4661.0,MYT1,HGNC:7622,Likely benign,0.0,"MeSH:D030342,MedGen:C0950123",...,T,T,T,T,T,64208182,CCGAGCGCTCCCAGGACCTGTGTCCCCAGTCCCTGGAGGATGCAGC...,CCGAGCGCTCCCAGGACCTGTGTCCCCAGTCCCTGGAGGATGCAGC...,EEEEEEEEEEEEEEEEEEEEEEEEEEEEAAPDVIFQEDTSHTSAQK...,EEEEEEEEEEEEEEEEEEEEEEEEEEEEAAPDVIFQEDTSHTSAQK...
6092,20_64208272_G_A,2367599.0,single nucleotide variant,NM_004535.3(MYT1):c.1076G>A (p.Arg359Gln),4661.0,MYT1,HGNC:7622,Likely benign,0.0,"MeSH:D030342,MedGen:C0950123",...,T,T,T,T,T,64208272,AGGAGGACGAGGAGGAGGAGGAGGAGGAAGAGGAGGAGGAGGAGGA...,AGGAGGACGAGGAGGAGGAGGAGGAGGAAGAGGAGGAGGAGGAGGA...,PDVIFQEDTSHTSAQKAPELRGPESPSPKPEYSVIVEVRSDDDKDE...,PDVIFQEDTSHTSAQKAPELRGPESPSPKPEYSVIVEVRSDDDKDE...
6093,20_64208481_A_G,742556.0,single nucleotide variant,NM_004535.3(MYT1):c.1285A>G (p.Ser429Gly),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900,...,T,T,T,T,T,64208481,CGGGGCCCAGAATCACCCAGTCCCAAGCCTGAGTACTCTGTTATTG...,CGGGGCCCAGAATCACCCAGTCCCAAGCCTGAGTACTCTGTTATTG...,GLLEQAIALKAEQVRTVCEPGCPPAEQSQLGLGEPGKAAKPLDTVR...,GLLEQAIALKAEQVRTVCEPGCPPAEQSQLGLGEPGKAAKPLDTVR...


In [7]:
PARQUET_PATH = r"D:\variant_data\clinvarhq_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\clinvarhq_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\clinvarhq_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\clinvarhq_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 703

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,CHROM,POS,REF,ALT,Label,ID,GeneInfo,CLNSIG,CLNREVSTAT,...,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,10_237638_G_T,chr10,237638,G,T,0,1980541,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,T,D,T,237638,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...
1,10_357852_G_C,chr10,357852,G,C,0,2758294,DIP2C,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,T,T,T,T,357852,TAAACACATTTTTCTTATGAAATTATATGGTCTCTTGCAGGAGAGG...,TAAACACATTTTTCTTATGAAATTATATGGTCTCTTGCAGGAGAGG...,CNVLMCPHTCVTNLPKPRQKQPEIGPASVMVGNLVSGKRIAQASGR...,CNVLMCPHTCVTNLPKPRQKQPEIGPASVMVGNLVSGKRIAQASGR...
2,10_1086292_C_T,chr10,1086292,C,T,0,2279214,WDR37,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,T,D,D,1086292,CGACCTCTGTCTGTCATTTTCCAGCATGATGCAGCCAGCCTGATAG...,CGACCTCTGTCTGTCATTTTCCAGCATGATGCAGCCAGCCTGATAG...,TSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPVVLGTA...,TSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPVVLGTA...
3,10_5102114_C_T,chr10,5102114,C,T,0,716665,AKR1C3,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,T,D,T,D,T,5102114,GTATCCCAGATATGGAACTTGTTACATCTCCTTCTAGTTGTCAAAG...,GTATCCCAGATATGGAACTTGTTACATCTCCTTCTAGTTGTCAAAG...,CTTWEAMEKCKDAGLAKSIGVSNFNRRQLEMILNKPGLKYKPVCNQ...,CTTWEAMEKCKDAGLAKSIGVSNFNRRQLEMILNKPGLKYKPVCNQ...
4,10_8064041_G_A,chr10,8064041,G,A,1,3384342,GATA3,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,T,D,D,8064041,TCTGTTGCAACGATGCATCTGCCCCTTCTGCGGGCGCCTCCGTGTG...,TCTGTTGCAACGATGCATCTGCCCCTTCTGCGGGCGCCTCCGTGTG...,VPEYSSGLFPPSSLLGGSPTGFGCKSRPKARSSTEGRECVNCGATS...,VPEYSSGLFPPSSLLGGSPTGFGCKSRPKARSSTEGRECVNCGATS...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
698,20_63488381_C_A,chr20,63488381,C,A,1,383531,EEF1A2,Likely_pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,D,D,D,63488381,GCGGGGGCGCCTTTCCTCTTGAAGAACTTCCACTGGACCTTGATGG...,GCGGGGGCGCCTTTCCTCTTGAAGAACTTCCACTGGACCTTGATGG...,LEDNPKSLKSGDAAIVEMVPGKPMCVESFSQYPPLGRFAVRDMRQT...,LEDNPKSLKSGDAAIVEMVPGKPMCVESFSQYPPLGRFAVRDMRQT...
699,20_63495909_C_T,chr20,63495909,C,T,1,279803,EEF1A2,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,D,D,D,63495909,CCCATCTATCTGGGGACTCTGACACTGGCTGGATGCCTTCACAGGC...,CCCATCTATCTGGGGACTCTGACACTGGCTGGATGCCTTCACAGGC...,KFEKEAAEMGKGSFKYAWVLDKLKAERERGITIDISLWKFETTKYY...,KFEKEAAEMGKGSFKYAWVLDKLKAERERGITIDISLWKFETTKYY...
700,20_63495972_C_T,chr20,63495972,C,T,1,100782,EEF1A2,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,D,D,D,63495972,AGCACTGGATTCATCCTTAGGGGGGCTCTGAGCCAGACTGGGTGAG...,AGCACTGGATTCATCCTTAGGGGGGCTCTGAGCCAGACTGGGTGAG...,KSTTTGHLIYKCGGIDKRTIEKFEKEAAEMGKGSFKYAWVLDKLKA...,KSTTTGHLIYKCGGIDKRTIEKFEKEAAEMGKGSFKYAWVLDKLKA...
701,20_63930873_T_G,chr20,63930873,T,G,1,30894,DNAJC5,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,D,D,T,D,D,63930873,TCCTGACCTCGTGATCCGCCAGCCTCGGCCTCCTGGAGTGCTGGGA...,TCCTGACCTCGTGATCCGCCAGCCTCGGCCTCCTGGAGTGCTGGGA...,AILTDATKRNIYDKYGSLGLYVAEQFGEENVNTYFVLSSWWAKALF...,AILTDATKRNIYDKYGSLGLYVAEQFGEENVNTYFVLSSWWAKALF...


In [8]:
PARQUET_PATH = r"D:\variant_data\proteingym_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\proteingym_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\proteingym_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\proteingym_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 1,472

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,CHROM,POS,REF,ALT,Label,protein,protein_sequence,mutant,mutated_sequence,...,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,10_180088_C_T,chr10,180088,C,T,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,R26W,MARLTKRRQADTKAIQHLWAAIEIIWNQKQIANIDRITKYMSRVHG...,...,D,D,D,T,T,180088,AATGTCATTCTTCATGATGTAATGAAATGAATGATACTTTATATGA...,AATGTCATTCTTCATGATGTAATGAAATGAATGATACTTTATATGA...,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...
1,10_240913_C_G,chr10,240913,C,G,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,C258W,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,D,D,D,T,240913,TCTGAAGTGCTAACCAGTGAGGTCCGGGCAGGGCCCAGTCAGACCA...,TCTGAAGTGCTAACCAGTGAGGTCCGGGCAGGGCCCAGTCAGACCA...,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...
2,10_242031_A_C,chr10,242031,A,C,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,H281P,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,D,D,D,D,242031,GCTGGCTCCCCAGCTGCACTTGGCCAAGCGGGTGCTCTTGTTTGCC...,GCTGGCTCCCCAGCTGCACTTGGCCAAGCGGGTGCTCTTGTTTGCC...,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...
3,10_248502_G_A,chr10,248502,G,A,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S465N,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,T,T,T,248502,AACTACATTTTTCAACAGCAGTTTATTCTATGGCTATTATGTATAT...,AACTACATTTTTCAACAGCAGTTTATTCTATGGCTATTATGTATAT...,TEAVSSSQEIPTMPQPIEKVSVSTQTKKLSASSPRMLHRSTQTTND...,TEAVSSSQEIPTMPQPIEKVSVSTQTKKLSASSPRMLHRSTQTTND...
4,10_248559_C_T,chr10,248559,C,T,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S484L,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,T,T,D,T,248559,TATCCGAATGGGGTAGTTCTTGAACTGGTGAATTATGTGGCTTCGT...,TATCCGAATGGGGTAGTTCTTGAACTGGTGAATTATGTGGCTTCGT...,VSVSTQTKKLSASSPRMLHRSTQTTNDGVCQSMCHDKYTKIFNDFK...,VSVSTQTKKLSASSPRMLHRSTQTTNDGVCQSMCHDKYTKIFNDFK...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1467,20_64207869_G_A,chr20,64207869,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,V225I,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,64207869,CCACGTTGATTTTGATTTTGTGCAGGAAGGAGCCCCGTCAAGTCCC...,CCACGTTGATTTTGATTTTGTGCAGGAAGGAGCCCCGTCAAGTCCC...,AEETLVEEDLGQAAKPGPGIVHLLQEAAEGAASEEGEKGLFIQPED...,AEETLVEEDLGQAAKPGPGIVHLLQEAAEGAASEEGEKGLFIQPED...
1468,20_64207986_G_C,chr20,64207986,G,C,1,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,E264Q,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,D,T,T,T,T,64207986,ATCGCAACTTCTCTCCTGAACTTGGGTCAAATTGCTGAAGAGACCC...,ATCGCAACTTCTCTCCTGAACTTGGGTCAAATTGCTGAAGAGACCC...,LFIQPEDAEEVVEVTTERSQDLCPQSLEDAASEESSKQKGILSHEE...,LFIQPEDAEEVVEVTTERSQDLCPQSLEDAASEESSKQKGILSHEE...
1469,20_64208061_G_A,chr20,64208061,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,E289K,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,64208061,AAGCCAGGTCCTGGCATTGTGCACCTGCTTCAGGAGGCTGCAGAGG...,AAGCCAGGTCCTGGCATTGTGCACCTGCTTCAGGAGGCTGCAGAGG...,SLEDAASEESSKQKGILSHEEEDEEEEEEEEEEEEDEEEEEEEEEE...,SLEDAASEESSKQKGILSHEEEDEEEEEEEEEEEEDEEEEEEEEEE...
1470,20_64208272_G_A,chr20,64208272,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,R359Q,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,64208272,AGGAGGACGAGGAGGAGGAGGAGGAGGAAGAGGAGGAGGAGGAGGA...,AGGAGGACGAGGAGGAGGAGGAGGAGGAAGAGGAGGAGGAGGAGGA...,PDVIFQEDTSHTSAQKAPELRGPESPSPKPEYSVIVEVRSDDDKDE...,PDVIFQEDTSHTSAQKAPELRGPESPSPKPEYSVIVEVRSDDDKDE...


In [9]:
PARQUET_PATH = r"D:\variant_data\uniprot_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\uniprot_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk['prot_ref_seq'], chunk['prot_alt_seq'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=['prot_ref_seq']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
            
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df['prot_ref_seq'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [101])") 

check_diff = df[df['prot_ref_seq'].str[50] == df['prot_alt_seq'].str[50]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\uniprot_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\uniprot_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 1,333

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [101] (Kỳ vọng: [101])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 50 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,CHROM,POS,REF,ALT,Label,dbSNP,gene,protein_AC,aa_change,...,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred,canonical_center,ref_seq,alt_seq,prot_ref_seq,prot_alt_seq
0,10_1072186_G_A,chr10,1072186,G,A,0,rs17856557,WDR37,Q9Y2I8,p.Ala11Thr,...,D,D,T,T,T,1072186,TGCCATTTGATACTTAAGATGTTAATGAAATTTGATAAAGAAATTA...,TGCCATTTGATACTTAAGATGTTAATGAAATTTGATAAAGAAATTA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMPTESA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMPTESA...
1,10_1096193_A_G,chr10,1096193,A,G,0,rs2306407,WDR37,Q9Y2I8,p.Ile225Val,...,T,T,T,T,T,1096193,ACATTTTCTCTGAACTTCCAGGACTTTCTATGCTAAAATGAAAGGT...,ACATTTTCTCTGAACTTCCAGGACTTTCTATGCTAAAATGAAAGGT...,ASADHTALLWSIETGKCLVKYAGHVGSVNSIKFHPSEQLALTASGD...,ASADHTALLWSIETGKCLVKYAGHVGSVNSIKFHPSEQLALTASGD...
2,10_1379131_C_A,chr10,1379131,C,A,0,rs3793733,ADARB2,Q9NS39,p.Ala44Thr,...,D,T,T,T,T,1379131,GGGTGAGGACAGGACCTCAAATAAGAAAAGGATTCCAGCTGAGGAC...,GGGTGAGGACAGGACCTCAAATAAGAAAAGGATTCCAGCTGAGGAC...,XXXXXXXMASVLGSGRGSGGLSSQLKCKSKRRRRRRSKRKDKVSIL...,XXXXXXXMASVLGSGRGSGGLSSQLKCKSKRRRRRRSKRKDKVSIL...
3,10_3151324_G_A,chr10,3151324,G,A,0,rs12248937,PITRM1,Q5JRX3,p.Ala554Asp,...,T,T,T,D,T,3151324,GAACACACACACACACGGGCTCTTACTGAGTGAAGGAGTCTATCAC...,GAACACACACACACACGGGCTCTTACTGAGTGAAGGAGTCTATCAC...,MRPDDKYHEKQAQVEATKLKQKVEALSPGDRQQIYEKGLELRSQQS...,MRPDDKYHEKQAQVEATKLKQKVEALSPGDRQQIYEKGLELRSQQS...
4,10_3165320_C_T,chr10,3165320,C,T,1,rs1249144069,PITRM1,Q5JRX3,p.Arg183Gln,...,D,D,T,D,D,3165320,ATACCAGGGGTCCCCAGACATGGTCTGAGACCTGCTGGGAAGAACA...,ATACCAGGGGTCCCCAGACATGGTCTGAGACCTGCTGGGAAGAACA...,TFMNAFTASDYTLYPFSTQNPKDFQNLLSVYLDATFFPCLRELDFW...,TFMNAFTASDYTLYPFSTQNPKDFQNLLSVYLDATFFPCLRELDFW...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,20_63495972_C_T,chr20,63495972,C,T,1,rs587777162,EEF1A2,Q05639,p.Gly70Ser,...,D,D,D,D,D,63495972,AGCACTGGATTCATCCTTAGGGGGGCTCTGAGCCAGACTGGGTGAG...,AGCACTGGATTCATCCTTAGGGGGGCTCTGAGCCAGACTGGGTGAG...,KSTTTGHLIYKCGGIDKRTIEKFEKEAAEMGKGSFKYAWVLDKLKA...,KSTTTGHLIYKCGGIDKRTIEKFEKEAAEMGKGSFKYAWVLDKLKA...
1329,20_63547241_C_A,chr20,63547241,C,A,0,rs55863722,SRMS,Q9H3Y6,p.Gly75Arg,...,D,D,T,D,D,63547241,ACCTGCCGGCCCGGGGCCCATCAGCTGTCCCTGAATGAATGCGTGG...,ACCTGCCGGCCCGGGGCCCATCAGCTGTCCCTGAATGAATGCGTGG...,EPDHGTPGSLDPNTDPVPTLPAEPCSPFPQLFLALYDFTARCGGEL...,EPDHGTPGSLDPNTDPVPTLPAEPCSPFPQLFLALYDFTARCGGEL...
1330,20_63547247_G_A,chr20,63547247,G,A,0,rs56053583,SRMS,Q9H3Y6,p.Arg73Cys,...,D,T,T,T,T,63547247,CGGCCCGGGGCCCATCAGCTGTCCCTGAATGAATGCGTGGATGAAC...,CGGCCCGGGGCCCATCAGCTGTCCCTGAATGAATGCGTGGATGAAC...,GGEPDHGTPGSLDPNTDPVPTLPAEPCSPFPQLFLALYDFTARCGG...,GGEPDHGTPGSLDPNTDPVPTLPAEPCSPFPQLFLALYDFTARCGG...
1331,20_63708763_C_A,chr20,63708763,C,A,0,rs1291212,ZGPAT,Q8N5A5,p.Ser61Arg,...,D,T,T,T,T,63708763,GGGAAAGGGGACGTGCCCGTGCCCGTGCCCGCCCTCAGGCTGTGGG...,GGGAAAGGGGACGTGCCCGTGCCCGTGCCCGCCCTCAGGCTGTGGG...,QTYRAQLQQVELALGAGLDSSEQADLRQLQGDLKELIELTEASLVS...,QTYRAQLQQVELALGAGLDSSEQADLRQLQGDLKELIELTEASLVS...
